# 17 — Computational Cost Analysis

**Goal:** Measure training time, inference latency, model size, and RAM usage for every MARS agent and the monolithic baseline. Results for **Section 4.5** of the paper.

**Two tables:**

| Table | Content |
|-------|---------|
| 1 | Per-agent cost: training time, inference time, model size, RAM |
| 2 | Pipeline latency: cold_start vs assessment vs continuous |

**Methodology:**
- Training time: wall-clock via `time.perf_counter()` during `train_all_agents`
- Inference time: average over 100 users, 100 repetitions for stable timing
- Model size: file sizes in `models/` + pickle serialization sizes
- RAM: `psutil.Process().memory_info()` before/after loading each agent
- Pipeline latency: time each pipeline call on test users

In [ ]:
import sys
import os
import time
import pickle
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")

# psutil for RAM measurement (with fallback)
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print("psutil not installed — RAM measurements will be skipped")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from agents.utils import set_global_seed, load_config

set_global_seed(42)

print(f"Project root: {ROOT}")
print(f"psutil available: {HAS_PSUTIL}")

## 1. Load Data & Config

In [ ]:
from data.loader import EdNetLoader

# Load splits
splits_dir = ROOT / "data" / "splits"
train_df = pd.read_parquet(splits_dir / "train.parquet")
val_df = pd.read_parquet(splits_dir / "val.parquet")
test_df = pd.read_parquet(splits_dir / "test.parquet")

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

# Load metadata
loader = EdNetLoader(data_dir=str(ROOT / "data" / "raw"))
questions_df = loader.questions
lectures_df = loader.lectures

print(f"Questions: {len(questions_df):,}  Lectures: {len(lectures_df):,}")

CONFIG = load_config(str(ROOT / "configs" / "config.yaml"))

# Sample test users for inference benchmarks
test_user_ids = test_df["user_id"].unique()
N_BENCH_USERS = min(100, len(test_user_ids))
bench_users = np.random.choice(test_user_ids, size=N_BENCH_USERS, replace=False)
print(f"Benchmark users: {N_BENCH_USERS}")

## 2. Helper Functions

In [ ]:
def get_process_memory_mb():
    """Current process RSS in MB."""
    if HAS_PSUTIL:
        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    return 0.0


def measure_model_size_mb(obj):
    """Estimate model size via pickle serialization (MB)."""
    try:
        data = pickle.dumps(obj)
        return len(data) / (1024 ** 2)
    except Exception:
        return float("nan")


def get_file_sizes_mb(directory, pattern="*"):
    """Sum file sizes in a directory matching pattern."""
    total = 0
    d = Path(directory)
    if d.exists():
        for f in d.rglob(pattern):
            if f.is_file():
                total += f.stat().st_size
    return total / (1024 ** 2)


def time_function(func, *args, n_reps=1, **kwargs):
    """Time a function call, return (result, elapsed_seconds)."""
    t0 = time.perf_counter()
    for _ in range(n_reps):
        result = func(*args, **kwargs)
    elapsed = (time.perf_counter() - t0) / n_reps
    return result, elapsed


print("Helpers defined.")

## 3. Per-Agent Training Time & Size Measurement

Train each agent individually, measuring wall-clock time and RAM delta.

In [ ]:
import gc
from agents.diagnostic_agent import DiagnosticAgent
from agents.confidence_agent import ConfidenceAgent
from agents.kg_agent import KnowledgeGraphAgent
from agents.prediction_agent import PredictionAgent
from agents.recommendation_agent import RecommendationAgent
from agents.personalization_agent import PersonalizationAgent, extract_user_features

set_global_seed(42)

# Storage for results
cost_data = {}
agent_objects = {}

AGENT_NAMES = [
    "Diagnostic (IRT)",
    "Confidence (XGBoost)",
    "KG (GraphSAGE)",
    "Prediction (LSTM)",
    "Recommendation (TS+LambdaMART)",
    "Personalization (K-Means)",
]

print("Starting per-agent training benchmarks...\n")

In [ ]:
# --- 1. DiagnosticAgent (IRT) ---
gc.collect()
mem_before = get_process_memory_mb()
t0 = time.perf_counter()

diag = DiagnosticAgent(seed=42)
irt_params = diag.calibrate_from_interactions(train_df, min_answers_per_q=20)

train_time_diag = time.perf_counter() - t0
mem_after = get_process_memory_mb()

cost_data["Diagnostic (IRT)"] = {
    "train_time_s": train_time_diag,
    "model_size_mb": measure_model_size_mb(diag),
    "ram_mb": mem_after - mem_before,
}
agent_objects["diagnostic"] = diag
print(f"Diagnostic: {train_time_diag:.1f}s, {cost_data['Diagnostic (IRT)']['model_size_mb']:.1f} MB")

In [ ]:
# --- 2. ConfidenceAgent (XGBoost) ---
gc.collect()
mem_before = get_process_memory_mb()
t0 = time.perf_counter()

conf = ConfidenceAgent()
conf_cfg = CONFIG.get("confidence", {})
conf.train(
    train_df,
    irt_params=irt_params,
    use_smote=conf_cfg.get("use_smote", False),
    use_class_weight=conf_cfg.get("use_class_weight", True),
)

train_time_conf = time.perf_counter() - t0
mem_after = get_process_memory_mb()

cost_data["Confidence (XGBoost)"] = {
    "train_time_s": train_time_conf,
    "model_size_mb": measure_model_size_mb(conf),
    "ram_mb": mem_after - mem_before,
}
agent_objects["confidence"] = conf
print(f"Confidence: {train_time_conf:.1f}s, {cost_data['Confidence (XGBoost)']['model_size_mb']:.1f} MB")

In [ ]:
# --- 3. KnowledgeGraphAgent (GraphSAGE) ---
gc.collect()
mem_before = get_process_memory_mb()
t0 = time.perf_counter()

kg = KnowledgeGraphAgent()
kg.build_graph(questions_df, lectures_df)
kg.build_prerequisites(train_df)
if hasattr(kg, "train_embeddings"):
    try:
        kg.train_embeddings()
    except Exception as e:
        print(f"  GraphSAGE embedding training: {e}")

train_time_kg = time.perf_counter() - t0
mem_after = get_process_memory_mb()

cost_data["KG (GraphSAGE)"] = {
    "train_time_s": train_time_kg,
    "model_size_mb": measure_model_size_mb(kg),
    "ram_mb": mem_after - mem_before,
}
agent_objects["knowledge_graph"] = kg
print(f"KG: {train_time_kg:.1f}s, {cost_data['KG (GraphSAGE)']['model_size_mb']:.1f} MB")

In [ ]:
# --- 4. PredictionAgent (LSTM) ---
gc.collect()
mem_before = get_process_memory_mb()
t0 = time.perf_counter()

pred = PredictionAgent()
pred.train(train_df, val_df=val_df)

train_time_pred = time.perf_counter() - t0
mem_after = get_process_memory_mb()

cost_data["Prediction (LSTM)"] = {
    "train_time_s": train_time_pred,
    "model_size_mb": measure_model_size_mb(pred),
    "ram_mb": mem_after - mem_before,
}
agent_objects["prediction"] = pred
print(f"Prediction: {train_time_pred:.1f}s, {cost_data['Prediction (LSTM)']['model_size_mb']:.1f} MB")

In [ ]:
# --- 5. RecommendationAgent (TS + LambdaMART) ---
gc.collect()
mem_before = get_process_memory_mb()
t0 = time.perf_counter()

rec = RecommendationAgent(random_seed=42)
if hasattr(rec, "initialize"):
    try:
        rec.initialize(
            questions_df=questions_df,
            lectures_df=lectures_df,
            interactions_df=train_df,
            train_user_ids=train_df["user_id"].unique().tolist(),
        )
    except Exception as e:
        print(f"  RecommendationAgent init: {e}")

train_time_rec = time.perf_counter() - t0
mem_after = get_process_memory_mb()

cost_data["Recommendation (TS+LambdaMART)"] = {
    "train_time_s": train_time_rec,
    "model_size_mb": measure_model_size_mb(rec),
    "ram_mb": mem_after - mem_before,
}
agent_objects["recommendation"] = rec
print(f"Recommendation: {train_time_rec:.1f}s, {cost_data['Recommendation (TS+LambdaMART)']['model_size_mb']:.1f} MB")

In [ ]:
# --- 6. PersonalizationAgent (K-Means) ---
gc.collect()
mem_before = get_process_memory_mb()
t0 = time.perf_counter()

pers = PersonalizationAgent()
user_feats = extract_user_features(train_df)
pers.train_clusters(user_feats)

train_time_pers = time.perf_counter() - t0
mem_after = get_process_memory_mb()

cost_data["Personalization (K-Means)"] = {
    "train_time_s": train_time_pers,
    "model_size_mb": measure_model_size_mb(pers),
    "ram_mb": mem_after - mem_before,
}
agent_objects["personalization"] = pers
print(f"Personalization: {train_time_pers:.1f}s, {cost_data['Personalization (K-Means)']['model_size_mb']:.1f} MB")

In [ ]:
# Check saved model file sizes on disk
models_dir = ROOT / "models"
for agent_dir_name in ["diagnostic", "confidence", "kg", "prediction", "recommendation", "personalization"]:
    agent_path = models_dir / agent_dir_name
    disk_mb = get_file_sizes_mb(agent_path)
    if disk_mb > 0:
        print(f"  {agent_dir_name:25s} disk: {disk_mb:.2f} MB")

print(f"\nTotal models/ directory: {get_file_sizes_mb(models_dir):.2f} MB")

## 4. Per-Agent Inference Time

Average inference latency over 100 benchmark users, 100 repetitions.

In [ ]:
from agents.orchestrator import Orchestrator

# Register all agents with orchestrator
orch = Orchestrator(seed=42)
for agent in agent_objects.values():
    orch.register_agent(agent)

print(f"Registered agents: {list(orch.agents.keys())}")

In [ ]:
N_REPS = 100

# Prepare per-user interaction data for benchmarking
def get_user_interactions(user_id):
    """Get a small interaction DataFrame for a benchmark user."""
    user_df = test_df[test_df["user_id"] == user_id].head(20)
    return user_df

def get_single_interaction(user_id):
    """Get a single interaction dict for continuous pipeline."""
    user_df = test_df[test_df["user_id"] == user_id].head(1)
    if len(user_df) == 0:
        return {"question_id": "q1", "correct": 1, "elapsed_time": 30000}
    row = user_df.iloc[0]
    return {
        "question_id": str(row.get("question_id", "q1")),
        "correct": int(row.get("correct", 1)),
        "elapsed_time": int(row.get("elapsed_time", 30000)),
    }

# --- Measure per-agent inference latency ---
print("Measuring per-agent inference latency...")

# DiagnosticAgent inference: update_theta for a set of responses
diag_agent = agent_objects["diagnostic"]
timings_diag = []
for uid in bench_users:
    user_df = get_user_interactions(uid)
    responses = []
    for _, row in user_df.head(10).iterrows():
        qid = str(row.get("question_id", ""))
        idx = diag_agent._qid_to_idx.get(qid)
        if idx is not None:
            responses.append((idx, bool(row["correct"])))
    if not responses:
        continue
    times = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        diag_agent.update_theta(responses)
        times.append(time.perf_counter() - t0)
    timings_diag.append(np.median(times))

cost_data["Diagnostic (IRT)"]["inference_ms"] = np.mean(timings_diag) * 1000 if timings_diag else float("nan")
print(f"  Diagnostic inference: {cost_data['Diagnostic (IRT)']['inference_ms']:.3f} ms")

In [ ]:
# ConfidenceAgent inference: classify_single
conf_agent = agent_objects["confidence"]
timings_conf = []
for uid in bench_users:
    interaction = get_single_interaction(uid)
    times = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        try:
            conf_agent.classify_single(user_id=str(uid), interaction=interaction)
        except Exception:
            pass
        times.append(time.perf_counter() - t0)
    timings_conf.append(np.median(times))

cost_data["Confidence (XGBoost)"]["inference_ms"] = np.mean(timings_conf) * 1000
print(f"  Confidence inference: {cost_data['Confidence (XGBoost)']['inference_ms']:.3f} ms")

In [ ]:
# KGAgent inference: get embeddings / profile lookup
kg_agent = agent_objects["knowledge_graph"]
timings_kg = []
for uid in bench_users:
    times = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        try:
            # Try common inference methods
            if hasattr(kg_agent, "handle_cold_start"):
                kg_agent.handle_cold_start(
                    user_id=str(uid),
                    diagnostic={"responses": []},
                    confidence={},
                )
            elif hasattr(kg_agent, "get_user_profile"):
                kg_agent.get_user_profile(str(uid))
        except Exception:
            pass
        times.append(time.perf_counter() - t0)
    timings_kg.append(np.median(times))

cost_data["KG (GraphSAGE)"]["inference_ms"] = np.mean(timings_kg) * 1000
print(f"  KG inference: {cost_data['KG (GraphSAGE)']['inference_ms']:.3f} ms")

In [ ]:
# PredictionAgent inference: predict / update_state
pred_agent = agent_objects["prediction"]
timings_pred = []
for uid in bench_users:
    interaction = get_single_interaction(uid)
    times = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        try:
            if hasattr(pred_agent, "update_state"):
                pred_agent.update_state(user_id=str(uid), interaction=interaction)
            elif hasattr(pred_agent, "predict_gaps"):
                pred_agent.predict_gaps(user_id=str(uid), diagnostic={}, kg_profile={})
        except Exception:
            pass
        times.append(time.perf_counter() - t0)
    timings_pred.append(np.median(times))

cost_data["Prediction (LSTM)"]["inference_ms"] = np.mean(timings_pred) * 1000
print(f"  Prediction inference: {cost_data['Prediction (LSTM)']['inference_ms']:.3f} ms")

In [ ]:
# RecommendationAgent inference: recommend
rec_agent = agent_objects["recommendation"]
timings_rec = []
for uid in bench_users:
    times = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        try:
            rec_agent.recommend(
                user_id=str(uid),
                kg_profile={},
                confidence={},
            )
        except Exception:
            pass
        times.append(time.perf_counter() - t0)
    timings_rec.append(np.median(times))

cost_data["Recommendation (TS+LambdaMART)"]["inference_ms"] = np.mean(timings_rec) * 1000
print(f"  Recommendation inference: {cost_data['Recommendation (TS+LambdaMART)']['inference_ms']:.3f} ms")

In [ ]:
# PersonalizationAgent inference: assign_cluster
pers_agent = agent_objects["personalization"]
timings_pers = []
for uid in bench_users:
    times = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        try:
            pers_agent.assign_cluster(
                user_id=str(uid),
                diagnostic={},
                confidence={},
            )
        except Exception:
            pass
        times.append(time.perf_counter() - t0)
    timings_pers.append(np.median(times))

cost_data["Personalization (K-Means)"]["inference_ms"] = np.mean(timings_pers) * 1000
print(f"  Personalization inference: {cost_data['Personalization (K-Means)']['inference_ms']:.3f} ms")

## 5. Table 1 — Per-Agent Computational Cost

In [ ]:
# Build Table 1
rows = []
for name in AGENT_NAMES:
    d = cost_data[name]
    rows.append({
        "Agent": name,
        "Training Time (min)": round(d["train_time_s"] / 60, 2),
        "Inference (ms/user)": round(d.get("inference_ms", float("nan")), 3),
        "Model Size (MB)": round(d["model_size_mb"], 2),
        "RAM Delta (MB)": round(d["ram_mb"], 1),
    })

# Totals for MARS
total_train = sum(cost_data[n]["train_time_s"] for n in AGENT_NAMES)
total_inference = sum(cost_data[n].get("inference_ms", 0) for n in AGENT_NAMES)
total_size = sum(cost_data[n]["model_size_mb"] for n in AGENT_NAMES)
total_ram = sum(cost_data[n]["ram_mb"] for n in AGENT_NAMES)

rows.append({
    "Agent": "**Total MARS**",
    "Training Time (min)": round(total_train / 60, 2),
    "Inference (ms/user)": round(total_inference, 3),
    "Model Size (MB)": round(total_size, 2),
    "RAM Delta (MB)": round(total_ram, 1),
})

table1_df = pd.DataFrame(rows)
print("Table 1: Per-Agent Computational Cost")
print("=" * 80)
print(table1_df.to_string(index=False))

## 6. Monolithic Baseline Cost

Train a single monolithic model (same pattern as notebook 13) and measure its cost for comparison.

In [ ]:
import torch
import torch.nn as nn

class MonolithicModel(nn.Module):
    """Single end-to-end model combining all agent functionality.
    Matches the architecture used in notebook 13 ablation study."""

    def __init__(self, n_questions, n_tags=188, embed_dim=64, hidden_dim=128, n_layers=2):
        super().__init__()
        self.q_embed = nn.Embedding(n_questions + 1, embed_dim, padding_idx=0)
        self.tag_embed = nn.Embedding(n_tags + 1, embed_dim, padding_idx=0)
        self.correct_embed = nn.Embedding(3, embed_dim // 2, padding_idx=0)  # 0=pad, 1=wrong, 2=correct

        input_dim = embed_dim * 2 + embed_dim // 2 + 1  # q + tag + correct + elapsed_norm
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True, dropout=0.2)

        # Multi-task heads
        self.head_correct = nn.Linear(hidden_dim, 1)        # next-correct prediction
        self.head_confidence = nn.Linear(hidden_dim, 4)      # 4-class confidence
        self.head_recommend = nn.Linear(hidden_dim, n_questions + 1)  # item scores

    def forward(self, q_ids, tag_ids, corrects, elapsed):
        q_emb = self.q_embed(q_ids)
        t_emb = self.tag_embed(tag_ids)
        c_emb = self.correct_embed(corrects)
        elapsed_norm = elapsed.unsqueeze(-1) / 300000.0
        x = torch.cat([q_emb, t_emb, c_emb, elapsed_norm], dim=-1)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return {
            "correct": torch.sigmoid(self.head_correct(last)),
            "confidence": self.head_confidence(last),
            "recommend": self.head_recommend(last),
        }


# Train monolithic model
n_questions = int(train_df["question_id"].nunique()) + 1

gc.collect()
mem_before_mono = get_process_memory_mb()
t0_mono = time.perf_counter()

mono_model = MonolithicModel(n_questions=n_questions)
optimizer = torch.optim.Adam(mono_model.parameters(), lr=1e-3)

# Simplified training loop (same as notebook 13 pattern)
mono_model.train()
n_epochs = 5
batch_size = 256

# Prepare sequences per user (simplified)
user_ids_train = train_df["user_id"].unique()
np.random.shuffle(user_ids_train)

for epoch in range(n_epochs):
    epoch_loss = 0.0
    n_batches = 0

    for batch_start in range(0, min(len(user_ids_train), 5000), batch_size):
        batch_uids = user_ids_train[batch_start:batch_start + batch_size]
        batch_dfs = [train_df[train_df["user_id"] == uid].head(50) for uid in batch_uids]
        batch_dfs = [df for df in batch_dfs if len(df) >= 3]

        if not batch_dfs:
            continue

        max_len = max(len(df) for df in batch_dfs)

        q_batch = torch.zeros(len(batch_dfs), max_len, dtype=torch.long)
        t_batch = torch.zeros(len(batch_dfs), max_len, dtype=torch.long)
        c_batch = torch.zeros(len(batch_dfs), max_len, dtype=torch.long)
        e_batch = torch.zeros(len(batch_dfs), max_len, dtype=torch.float)
        target_batch = torch.zeros(len(batch_dfs), dtype=torch.float)

        for i, df in enumerate(batch_dfs):
            seq_len = len(df)
            # Use hash-based encoding for question IDs
            q_ids = torch.tensor([abs(hash(str(qid))) % n_questions + 1 for qid in df["question_id"].values], dtype=torch.long)
            q_batch[i, :seq_len] = q_ids
            if "tags" in df.columns:
                t_batch[i, :seq_len] = torch.tensor([abs(hash(str(t))) % 188 + 1 for t in df["tags"].values], dtype=torch.long)
            c_batch[i, :seq_len] = torch.tensor(df["correct"].values + 1, dtype=torch.long)  # shift: 1=wrong, 2=correct
            if "elapsed_time" in df.columns:
                e_batch[i, :seq_len] = torch.tensor(df["elapsed_time"].values, dtype=torch.float)
            target_batch[i] = float(df["correct"].iloc[-1])

        optimizer.zero_grad()
        output = mono_model(q_batch, t_batch, c_batch, e_batch)
        loss = nn.functional.binary_cross_entropy(output["correct"].squeeze(), target_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    if n_batches > 0:
        print(f"  Mono epoch {epoch+1}/{n_epochs}  loss={epoch_loss/n_batches:.4f}")

train_time_mono = time.perf_counter() - t0_mono
mem_after_mono = get_process_memory_mb()

mono_model.eval()

print(f"\nMonolithic training: {train_time_mono:.1f}s ({train_time_mono/60:.2f} min)")

In [ ]:
# Monolithic inference latency
timings_mono = []
for uid in bench_users:
    user_df = test_df[test_df["user_id"] == uid].head(50)
    if len(user_df) < 3:
        continue

    seq_len = len(user_df)
    q_ids = torch.tensor([[abs(hash(str(qid))) % n_questions + 1 for qid in user_df["question_id"].values]], dtype=torch.long)
    t_ids = torch.zeros(1, seq_len, dtype=torch.long)
    if "tags" in user_df.columns:
        t_ids = torch.tensor([[abs(hash(str(t))) % 188 + 1 for t in user_df["tags"].values]], dtype=torch.long)
    c_ids = torch.tensor([user_df["correct"].values + 1], dtype=torch.long)
    e_vals = torch.zeros(1, seq_len, dtype=torch.float)
    if "elapsed_time" in user_df.columns:
        e_vals = torch.tensor([user_df["elapsed_time"].values], dtype=torch.float)

    times = []
    with torch.no_grad():
        for _ in range(N_REPS):
            t0 = time.perf_counter()
            mono_model(q_ids, t_ids, c_ids, e_vals)
            times.append(time.perf_counter() - t0)
    timings_mono.append(np.median(times))

mono_inference_ms = np.mean(timings_mono) * 1000 if timings_mono else float("nan")
mono_size_mb = measure_model_size_mb(mono_model)
mono_ram_mb = mem_after_mono - mem_before_mono

print(f"Monolithic inference: {mono_inference_ms:.3f} ms")
print(f"Monolithic model size: {mono_size_mb:.2f} MB")
print(f"Monolithic RAM delta: {mono_ram_mb:.1f} MB")

In [ ]:
# Add monolithic row to table 1
rows.append({
    "Agent": "Monolithic baseline",
    "Training Time (min)": round(train_time_mono / 60, 2),
    "Inference (ms/user)": round(mono_inference_ms, 3),
    "Model Size (MB)": round(mono_size_mb, 2),
    "RAM Delta (MB)": round(mono_ram_mb, 1),
})

table1_df = pd.DataFrame(rows)
print("\nTable 1: Per-Agent Computational Cost (with Monolithic)")
print("=" * 85)
print(table1_df.to_string(index=False))

## 7. Pipeline Latency (Table 2)

Measure end-to-end latency for each orchestrator pipeline.

In [ ]:
# Measure pipeline latencies
pipeline_latencies = defaultdict(list)

print("Measuring pipeline latencies...")

for uid in bench_users[:N_BENCH_USERS]:
    uid_str = str(uid)
    user_interactions = get_user_interactions(uid)
    single_interaction = get_single_interaction(uid)

    # --- cold_start pipeline ---
    try:
        t0 = time.perf_counter()
        orch.cold_start_pipeline(uid_str)
        pipeline_latencies["cold_start"].append((time.perf_counter() - t0) * 1000)
    except Exception as e:
        if len(pipeline_latencies["cold_start"]) == 0:
            print(f"  cold_start error: {e}")

    # --- assessment pipeline ---
    try:
        t0 = time.perf_counter()
        orch.assessment_pipeline(uid_str, user_interactions)
        pipeline_latencies["assessment"].append((time.perf_counter() - t0) * 1000)
    except Exception as e:
        if len(pipeline_latencies["assessment"]) == 0:
            print(f"  assessment error: {e}")

    # --- continuous pipeline ---
    try:
        t0 = time.perf_counter()
        orch.continuous_pipeline(uid_str, single_interaction)
        pipeline_latencies["continuous"].append((time.perf_counter() - t0) * 1000)
    except Exception as e:
        if len(pipeline_latencies["continuous"]) == 0:
            print(f"  continuous error: {e}")

for name, times in pipeline_latencies.items():
    if times:
        print(f"  {name:15s}: mean={np.mean(times):.1f} ms, median={np.median(times):.1f} ms, std={np.std(times):.1f} ms (n={len(times)})")

In [ ]:
# Build Table 2: Pipeline Latency
pipeline_info = {
    "cold_start":  {"agents_called": "all 6 (Diag, Conf, KG, Rec, Pers)",  "n_agents": 5},
    "assessment":  {"agents_called": "all 6 (Diag, Conf, KG, Pred, Rec, Pers)", "n_agents": 6},
    "continuous":  {"agents_called": "3-4 (Conf, Diag, Pred, [Rec])",       "n_agents": 3},
}

table2_rows = []
for pipe_name in ["cold_start", "assessment", "continuous"]:
    times = pipeline_latencies.get(pipe_name, [])
    info = pipeline_info[pipe_name]
    note = "" if pipe_name != "continuous" else " <-- main MAS advantage"
    table2_rows.append({
        "Pipeline": pipe_name,
        "Agents Called": info["agents_called"],
        "N Agents": info["n_agents"],
        "Mean Latency (ms)": round(np.mean(times), 1) if times else float("nan"),
        "Median Latency (ms)": round(np.median(times), 1) if times else float("nan"),
        "Std (ms)": round(np.std(times), 1) if times else float("nan"),
        "Note": note,
    })

table2_df = pd.DataFrame(table2_rows)
print("\nTable 2: Pipeline Latency")
print("=" * 100)
print(table2_df.to_string(index=False))

# Speedup of continuous vs cold_start
if pipeline_latencies.get("cold_start") and pipeline_latencies.get("continuous"):
    speedup = np.mean(pipeline_latencies["cold_start"]) / np.mean(pipeline_latencies["continuous"])
    print(f"\nSpeedup (cold_start / continuous): {speedup:.1f}x")

## 8. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Pipeline Latency Bar Chart ---
ax = axes[0]
pipe_names = ["cold_start", "assessment", "continuous"]
pipe_means = [np.mean(pipeline_latencies.get(p, [0])) for p in pipe_names]
pipe_stds = [np.std(pipeline_latencies.get(p, [0])) for p in pipe_names]

colors_pipe = ["#e74c3c", "#f39c12", "#27ae60"]
bars = ax.bar(pipe_names, pipe_means, yerr=pipe_stds, capsize=5,
              color=colors_pipe, edgecolor="black", linewidth=0.8, alpha=0.85)

for bar, val in zip(bars, pipe_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(pipe_stds)*0.3,
            f"{val:.1f} ms", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("Latency (ms)", fontsize=11)
ax.set_title("Pipeline Latency Comparison", fontsize=12, fontweight="bold")
ax.set_xlabel("Pipeline", fontsize=11)
ax.grid(axis="y", alpha=0.3)

# --- Plot 2: MARS vs Monolithic Training Time ---
ax = axes[1]
mars_min = total_train / 60
mono_min = train_time_mono / 60

bars2 = ax.bar(["MARS\n(6 agents)", "Monolithic"], [mars_min, mono_min],
               color=["#3498db", "#95a5a6"], edgecolor="black", linewidth=0.8, alpha=0.85)

for bar, val in zip(bars2, [mars_min, mono_min]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f"{val:.1f} min", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("Training Time (min)", fontsize=11)
ax.set_title("Training Time: MARS vs Monolithic", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# --- Plot 3: Training Time Distribution (Pie) ---
ax = axes[2]
agent_times = [cost_data[n]["train_time_s"] for n in AGENT_NAMES]
agent_short = ["IRT", "XGBoost", "GraphSAGE", "LSTM", "TS+LM", "K-Means"]
colors_pie = ["#e74c3c", "#f39c12", "#27ae60", "#3498db", "#9b59b6", "#1abc9c"]

def autopct_func(pct):
    return f"{pct:.1f}%" if pct > 5 else ""

wedges, texts, autotexts = ax.pie(
    agent_times, labels=agent_short, colors=colors_pie,
    autopct=autopct_func, startangle=90,
    textprops={"fontsize": 9},
)
ax.set_title("Training Time Distribution", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(str(ROOT / "results" / "computational_cost" / "cost_comparison.png"), dpi=150, bbox_inches="tight")
plt.savefig(str(ROOT / "results" / "computational_cost" / "cost_comparison.pdf"), bbox_inches="tight")
plt.show()
print("Figures saved.")

## 9. Save Results & LaTeX Tables

In [ ]:
# Create output directory
out_dir = ROOT / "results" / "computational_cost"
out_dir.mkdir(parents=True, exist_ok=True)

# Save raw data as JSON
results_json = {
    "per_agent": {},
    "monolithic": {
        "train_time_s": train_time_mono,
        "inference_ms": mono_inference_ms,
        "model_size_mb": mono_size_mb,
        "ram_mb": mono_ram_mb,
    },
    "pipeline_latencies": {
        name: {
            "mean_ms": float(np.mean(times)),
            "median_ms": float(np.median(times)),
            "std_ms": float(np.std(times)),
            "n_samples": len(times),
        }
        for name, times in pipeline_latencies.items()
    },
}

for name in AGENT_NAMES:
    results_json["per_agent"][name] = {
        k: float(v) if isinstance(v, (np.floating, float)) else v
        for k, v in cost_data[name].items()
    }

with open(out_dir / "computational_cost.json", "w") as f:
    json.dump(results_json, f, indent=2, default=str)

# Save tables as CSV
table1_df.to_csv(out_dir / "table1_per_agent_cost.csv", index=False)
table2_df.to_csv(out_dir / "table2_pipeline_latency.csv", index=False)

print(f"Results saved to {out_dir}")
print(f"  - computational_cost.json")
print(f"  - table1_per_agent_cost.csv")
print(f"  - table2_pipeline_latency.csv")

In [ ]:
# --- LaTeX Table 1: Per-Agent Cost ---
print("% LaTeX Table 1: Per-Agent Computational Cost")
print(r"\begin{table}[htbp]")
print(r"\centering")
print(r"\caption{Per-agent computational cost of MARS.}")
print(r"\label{tab:per-agent-cost}")
print(r"\begin{tabular}{lrrrr}")
print(r"\toprule")
print(r"Agent & Train (min) & Inference (ms) & Size (MB) & RAM (MB) \\")
print(r"\midrule")

for _, row in table1_df.iterrows():
    name = row["Agent"].replace("**", "").replace("_", r"\_")
    if "Total MARS" in name:
        print(r"\midrule")
        name = r"\textbf{Total MARS}"
    elif "Monolithic" in name:
        name = r"Monolithic baseline"
    train_min = row["Training Time (min)"]
    inf_ms = row["Inference (ms/user)"]
    size_mb = row["Model Size (MB)"]
    ram_mb = row["RAM Delta (MB)"]
    print(f"{name} & {train_min} & {inf_ms} & {size_mb} & {ram_mb} \\\\")

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

# Save to file
with open(out_dir / "table1_per_agent_cost.tex", "w", encoding="utf-8") as f:
    f.write("% Per-Agent Computational Cost\n")
    f.write(r"\begin{table}[htbp]" + "\n")
    f.write(r"\centering" + "\n")
    f.write(r"\caption{Per-agent computational cost of MARS.}" + "\n")
    f.write(r"\label{tab:per-agent-cost}" + "\n")
    f.write(r"\begin{tabular}{lrrrr}" + "\n")
    f.write(r"\toprule" + "\n")
    f.write(r"Agent & Train (min) & Inference (ms) & Size (MB) & RAM (MB) \\" + "\n")
    f.write(r"\midrule" + "\n")
    for _, row in table1_df.iterrows():
        name = row["Agent"].replace("**", "").replace("_", r"\_")
        if "Total MARS" in name:
            f.write(r"\midrule" + "\n")
            name = r"\textbf{Total MARS}"
        elif "Monolithic" in name:
            name = r"Monolithic baseline"
        f.write(f"{name} & {row['Training Time (min)']} & {row['Inference (ms/user)']} & {row['Model Size (MB)']} & {row['RAM Delta (MB)']} \\\\\n")
    f.write(r"\bottomrule" + "\n")
    f.write(r"\end{tabular}" + "\n")
    f.write(r"\end{table}" + "\n")

print(f"\nSaved: {out_dir / 'table1_per_agent_cost.tex'}")

In [ ]:
# --- LaTeX Table 2: Pipeline Latency ---
print("% LaTeX Table 2: Pipeline Latency")
print(r"\begin{table}[htbp]")
print(r"\centering")
print(r"\caption{Pipeline latency comparison in MARS.}")
print(r"\label{tab:pipeline-latency}")
print(r"\begin{tabular}{llrr}")
print(r"\toprule")
print(r"Pipeline & Agents Called & Mean Latency (ms) & Std (ms) \\")
print(r"\midrule")

for _, row in table2_df.iterrows():
    pipe = row["Pipeline"].replace("_", r"\_")
    agents = row["Agents Called"]
    mean_lat = row["Mean Latency (ms)"]
    std_lat = row["Std (ms)"]
    print(f"{pipe} & {agents} & {mean_lat} & {std_lat} \\\\")

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

# Save to file
with open(out_dir / "table2_pipeline_latency.tex", "w", encoding="utf-8") as f:
    f.write("% Pipeline Latency Comparison\n")
    f.write(r"\begin{table}[htbp]" + "\n")
    f.write(r"\centering" + "\n")
    f.write(r"\caption{Pipeline latency comparison in MARS.}" + "\n")
    f.write(r"\label{tab:pipeline-latency}" + "\n")
    f.write(r"\begin{tabular}{llrr}" + "\n")
    f.write(r"\toprule" + "\n")
    f.write(r"Pipeline & Agents Called & Mean Latency (ms) & Std (ms) \\" + "\n")
    f.write(r"\midrule" + "\n")
    for _, row in table2_df.iterrows():
        pipe = row["Pipeline"].replace("_", r"\_")
        agents = row["Agents Called"]
        f.write(f"{pipe} & {agents} & {row['Mean Latency (ms)']} & {row['Std (ms)']} \\\\\n")
    f.write(r"\bottomrule" + "\n")
    f.write(r"\end{tabular}" + "\n")
    f.write(r"\end{table}" + "\n")

print(f"\nSaved: {out_dir / 'table2_pipeline_latency.tex'}")

## 10. Summary for Section 4.5

Key findings to report in the paper.

In [ ]:
print("=" * 70)
print("SUMMARY FOR SECTION 4.5: COMPUTATIONAL COST")
print("=" * 70)

print(f"\n--- Training ---")
print(f"Total MARS training time:      {total_train/60:.2f} min")
print(f"Monolithic training time:       {train_time_mono/60:.2f} min")
train_ratio = total_train / train_time_mono if train_time_mono > 0 else float("nan")
print(f"Ratio (MARS / Monolithic):      {train_ratio:.2f}x")

print(f"\nMost expensive agent:           {AGENT_NAMES[np.argmax(agent_times)]}")
print(f"  -> {max(agent_times)/60:.2f} min ({max(agent_times)/total_train*100:.1f}% of total)")

print(f"\n--- Inference ---")
print(f"Total MARS inference (all agents): {total_inference:.3f} ms")
print(f"Monolithic inference:              {mono_inference_ms:.3f} ms")

print(f"\n--- Pipeline Latency ---")
for pipe_name in ["cold_start", "assessment", "continuous"]:
    times = pipeline_latencies.get(pipe_name, [])
    if times:
        print(f"{pipe_name:15s}: {np.mean(times):.1f} ms (median: {np.median(times):.1f} ms)")

if pipeline_latencies.get("cold_start") and pipeline_latencies.get("continuous"):
    speedup = np.mean(pipeline_latencies["cold_start"]) / np.mean(pipeline_latencies["continuous"])
    print(f"\nContinuous pipeline is {speedup:.1f}x faster than cold_start")
    print("-> This is the main MAS advantage: not all agents are needed for every interaction.")

print(f"\n--- Model Size ---")
print(f"Total MARS model size: {total_size:.2f} MB")
print(f"Monolithic model size: {mono_size_mb:.2f} MB")

print(f"\n--- RAM ---")
if HAS_PSUTIL:
    print(f"Total MARS RAM delta: {total_ram:.1f} MB")
    print(f"Monolithic RAM delta: {mono_ram_mb:.1f} MB")
else:
    print("(psutil not available, RAM not measured)")

print("\n" + "=" * 70)